# Step 11: Re-verify checkpoints affected by the tie_word_embeddings bug (Colab GPU)

**Bug found and fixed today:** two local checkpoints -- Model 1 variant 2
(`model1_reactant_v2/final`) and Model 2's ReactionT5-base run
(`model2_conditions_reactiont5base/final`) -- have `config.json`'s `tie_word_embeddings`
wrongly set to `true`, when the actual saved weights have *distinct* `lm_head.weight` and
`shared.weight` tensors. This makes generation silently degenerate into repeating one token
(confirmed directly: near-0% valid-SMILES rate) with the currently-installed `transformers`
version -- it isn't caught as a loud crash, just garbage output, which is why it went unnoticed
until a suspicious result (a fine-tuned checkpoint scoring *below* its own unfine-tuned starting
point) triggered a deeper investigation.

**Both `run_reactiont5_topk.py` and `evaluate_conditions_model_topk.py` now auto-detect and fix
this from the actual saved weights on load** (compares `lm_head.weight` vs `shared.weight`
directly, doesn't just trust the config field) -- `git pull` below already has the fix, no manual
step needed beyond uploading the checkpoints.

**Checkpoints affected by this bug are NOT affected:** every Kaggle-trained Model 1 checkpoint
(150k, 300k, root-aligned) already had `tie_word_embeddings=false` correctly set -- verified
directly, not assumed. Model 2's original `t5-small` run is also unaffected (genuinely tied,
no separate `lm_head.weight` tensor was ever saved for it). This notebook only needs to re-check
the two specifically-confirmed-broken checkpoints.

**Before running:** in the notebook Settings panel (right sidebar) turn on **Internet** and **GPU
accelerator** (T4).

**Checkpoint uploads required** (both from `~/Documents/diploma/retro-planner-checkpoints/`):
- `model1_reactant_v2/final` (~758MB)
- `model2_conditions_reactiont5base/final` (~758MB)

Upload both to Google Drive (drag-and-drop on drive.google.com), then set the paths below.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
import os

if not os.path.isdir("retro-planner"):
    !git clone https://github.com/oleh-kuzmenko/retro-planner.git
%cd retro-planner
!git pull

In [ ]:
%pip install -q -e ".[local-models,indexing]"

In [ ]:
v2_checkpoint_path = "/content/drive/MyDrive/retro-planner-checkpoints/model1_reactant_v2/final"  # @param {type:"string"}
model2_reactiont5base_checkpoint_path = "/content/drive/MyDrive/retro-planner-checkpoints/model2_conditions_reactiont5base/final"  # @param {type:"string"}

import os
for p in (v2_checkpoint_path, model2_reactiont5base_checkpoint_path):
    assert os.path.isdir(p), f"Not found: {p} -- did you upload it to Drive and set the path above?"
print("Both checkpoints found.")

### Model 1 variant 2, re-verified (ORD + USPTO)

Historical numbers to compare against (`RESULTS.md`): ORD exact_match top-1=50.7%, core=62.3%,
top-5 exact=74.3%, core=79.7%; USPTO exact=21.7%, core=43.3%. A local 5-record smoke test with the
fix already showed 80% top-1 exact / 100% top-1 core -- consistent with the historical result, not
the broken ~0%.

In [ ]:
!python scripts/models/run_reactiont5_topk.py \
    --input data/v2_ord_eval_targets.json \
    --t5-model "{v2_checkpoint_path}" \
    --num-beams 10 --device cuda --batch-size 32 \
    --output experiments/v2_model1_topk/v2_refixed_ord_topk.json

In [ ]:
!python scripts/models/run_reactiont5_topk.py \
    --input data/v2_uspto_eval_targets.json \
    --t5-model "{v2_checkpoint_path}" \
    --num-beams 10 --device cuda --batch-size 32 \
    --output experiments/v2_model1_topk/v2_refixed_uspto_topk.json

### Model 2, ReactionT5-base, re-verified (full 2285-record ORD conditions test set)

This is the run that was killed after 5.5+ CPU-hours locally once the bug was found -- rerun here
with the fix, on GPU, should take a fraction of the time.

In [ ]:
!python scripts/evaluate_conditions_model_topk.py \
    --model-dir "{model2_reactiont5base_checkpoint_path}" \
    --test-file data/v2_ord_train/conditions_test.jsonl \
    --num-beams 10 --device cuda \
    --output experiments/v2_model2_topk/conditions_reactiont5base_topk.json

**Note:** `evaluate_conditions_model_topk.py` doesn't have a `--batch-size` flag (it wasn't part of
the same speed fix as `run_reactiont5_topk.py`) -- it'll still be much faster than CPU purely from
the GPU forward-pass speedup, just not from batching.

**When everything finishes:** download the four result files (`v2_refixed_ord_topk.json`,
`v2_refixed_uspto_topk.json`, `conditions_reactiont5base_topk.json`) back to the local repo's
matching `experiments/` folders so RESULTS.md can be updated.